In [1]:
SUMMARY_STATS_FIELDS = [
    "baseApr",
    "feeApr",
    "incentiveApr",
    "safeTotalSupply",
    "priceReturn",
    "maxDiscount",  # not used
    "maxPremium",  # not used
    "ownedShares",
    "compositeReturn",
    "pricePerShare",
    "pointsApr",
]

from mainnet_launch.pages.solver_diagnostics.solver_diagnostics import (
    _load_solver_df,
    ensure_all_rebalance_plans_are_loaded_from_s3_bucket,
    SOLVER_REBALANCE_PLANS_DIR,
    AUTO_ETH,
    load_solver_plans,
)
import pandas as pd
import json
from mainnet_launch.constants import ALL_AUTOPOOLS

ensure_all_rebalance_plans_are_loaded_from_s3_bucket()

autopool_plans = [str(p) for p in SOLVER_REBALANCE_PLANS_DIR.glob("*.json")]
a = autopool_plans[0].split("/")[-1].split("_")
timestamp, autopool_address = a[2], a[3].replace(".json", "")
records = []

for p in SOLVER_REBALANCE_PLANS_DIR.glob("*.json"):
    a = str(p).split("/")[-1].split("_")
    timestamp, autopool_address = a[2], a[3].replace(".json", "")
    if autopool_address in [v.autopool_addr for v in ALL_AUTOPOOLS]:
        records.append({"timestamp": int(timestamp), "autopool_address": autopool_address})

df = pd.DataFrame.from_records(records)

2025-04-09 13:30:19.771 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [ ]:
from mainnet_launch.data_fetching.fetch_block_utils import get_nearest_blocks_for_timestamps, ETH_CHAIN

timestamps = df["timestamp"].to_list()[:1000]

blocks = get_nearest_blocks_for_timestamps(timestamps, ETH_CHAIN)
# good enough
blocks

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
208
209
210
211
212
213
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273


{1729575033: 21019041,
 1732771814: 21284146,
 1733689814: 21360173,
 1742218221: 22066919,
 1726984847: 20804186,
 1741167030: 21979799,
 1741780840: 22030648,
 1735933590: 21546134,
 1733201190: 21319679,
 1740659430: 21937684,
 1731730590: 21197808,
 1739032221: 21802997,
 1732977990: 21301179,
 1733542230: 21347941,
 1741338040: 21993968,
 1732756590: 21282903,
 1739554214: 21846197,
 1737031590: 21637135,
 1743777027: 22196162,
 1741663830: 22020957,
 1733013990: 21304165,
 1743942602: 22209878,
 1728531850: 20932598,
 1734990390: 21467954,
 1734633990: 21438446,
 1734576390: 21433674,
 1741764621: 22029304,
 1729542633: 21016357,
 1735101990: 21477191,
 1741878040: 22038704,
 1737286214: 21658243,
 1736832614: 21620640,
 1738377013: 21748695,
 1742536840: 22093311,
 1731685590: 21194079,
 1732090590: 21227697,
 1736855190: 21622516,
 1740036621: 21886088,
 1740101421: 21891441,
 1730140213: 21065929,
 1742977827: 22129892,
 1731081135: 21143937,
 1736754390: 21614164,
 1735749021

In [3]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

In [4]:
import requests


def fetch_blocks_by_timestamps(timestamps, graphql_url):
    """
    Fetch blocks from a GraphQL endpoint where the block timestamp is in the provided list.

    Parameters:
      timestamps (list[int]): A list of timestamp values.
      graphql_url (str): The URL of the GraphQL endpoint.

    Returns:
      list[dict]: A list of block objects matching the filter, each with id, number, and timestamp.
    """
    page_size = 1000  # Fetch up to 1000 items per page
    skip = 0
    all_blocks = []

    query = """
    query($timestamps: [Int!], $first: Int!, $skip: Int!) {
      blocks(
        where: { timestamp_in: $timestamps }
        first: $first
        skip: $skip
      ) {
        id
        number
        timestamp
      }
    }
    """

    while True:
        variables = {"timestamps": timestamps, "first": page_size, "skip": skip}

        response = requests.post(graphql_url, json={"query": query, "variables": variables})
        if response.status_code != 200:
            raise Exception(f"GraphQL query failed with status {response.status_code}: {response.text}")

        data = response.json()
        # Check for any GraphQL errors
        if "errors" in data:
            raise Exception(f"GraphQL errors returned: {data['errors']}")

        blocks = data.get("data", {}).get("blocks", [])
        if not blocks:
            # No more blocks to fetch, exit the loop
            break

        all_blocks.extend(blocks)
        if len(blocks) < page_size:
            # Last page received
            break

        skip += page_size

    return all_blocks


# Example usage:

# Define your list of timestamps
timestamps = [1726789351, 1726789350]  # Example timestamps
# Replace with your actual GraphQL endpoint URL
graphql_url = "https://subgraph.satsuma-prod.com/community/blocks-base/api"
blocks = fetch_blocks_by_timestamps(timestamps, graphql_url)

Exception: GraphQL query failed with status 403: 

In [ ]:
df["block"].value_counts()

In [ ]:
from mainnet_launch.constants import AUTO_USD, AUTO_ETH

autoETH_plans = load_solver_plans(AUTO_ETH)
autoUSD_plans = load_solver_plans(AUTO_USD)

In [ ]:
import pandas as pd


def normalize_plan(plan: dict) -> list:
    """
    Normalizes the plan JSON into a list of summary dictionaries suitable for conversion
    into a pandas DataFrame.

    For each destination state (destState) in the plan, the function extracts and computes
    the following fields:

      - name: The pool name.
      - vault address: The pool's address.
      - incentiveApr: The incentive APR as provided.
      - ownedShares: Normalized (if stored as a large integer) using the value in 'decimals'
                     (i.e. int(ownedShares) / (10 ** decimals)).
      - compositeReturn: Computed as the average of totalAprIn and totalAprOut.
      - pricePerShare: Taken from 'safePrice' (or optionally one could use 'spotPrice').

    Parameters:
        plan (dict): The plan JSON data.

    Returns:
        list: A list of dictionaries with the normalized summary statistics fields.
    """
    normalized_list = []
    # Retrieve destination states list from the "sod" section using direct lookups.
    dest_states = plan["sod"]["destStates"]

    for dest in dest_states:
        # Extract the decimals and use the first element.
        if "decimals" not in dest:
            decimals = 18
        else:
            # autoUSD
            decimals = dest["decimals"][0]

        # Normalize the ownedShares field using the decimals value.
        owned_shares_raw = dest["ownedShares"]
        normalized_owned_shares = int(owned_shares_raw) / (10**decimals)

        # Compute compositeReturn as the average of totalAprIn and totalAprOut.
        total_apr_in = dest["totalAprIn"]
        total_apr_out = dest["totalAprOut"]

        # Retrieve safePrice as pricePerShare.
        price_per_share = dest["safePrice"]

        lp_token_total_supply = dest["totSupply"] / 1e18

        # Build the normalized entry.
        normalized_entry = {
            "name": dest["name"],
            "vault address": dest["address"],
            "incentiveApr": dest["incentiveAPR"],
            "ownedShares": normalized_owned_shares,
            "total_apr_in": total_apr_in,
            "total_apr_out": total_apr_out,
            "pricePerShare": price_per_share,
            "underlyingReserves": [r / 10**decimals for r in dest["underlyingReserves"]],
            "lp_token_total_supply": lp_token_total_supply,
        }
        normalized_list.append(normalized_entry)

    return normalized_list


pd.DataFrame.from_records(normalize_plan(autoETH_plans[-1]))

In [ ]:
pd.DataFrame.from_records(normalize_plan(autoUSD_plans[-1]))

In [ ]:
fields_not_to_modify = ["name,"]

dict_keys(
    [
        "snapshotTimestamp",
        "name",
        "address",
        "poolType",
        "pool",
        "underlying",
        "underlyingTokens",
        "underlyingTokenAmounts",
        "underlyingReserves",
        "totalAprIn",
        "totalAprOut",
        "incentiveAPR",
        "destLPValue",
        "discountViolationAddFlag",
        "discountViolationTrim1Flag",
        "discountViolationTrim2Flag",
        "ownedShares",
        "totSupply",
        "safePrice",
        "spotPrice",
        "tokenSpotPrice",
        "tokenSafePrice",
        "flipFlag",
    ]
)

In [ ]:
autoETH_plans[-1]["sod"]["destStates"][0]

In [ ]:
autoETH_plans[-1]["sod"]["destStates"][0].keys()

In [ ]:
autoUSD_plans[-1]

In [ ]:
autoUSD_plans[-1]["sod"]["destStates"][0].keys()

In [ ]:
def _get_destination_states(plan: dict):
    if "sod" in plan and "destStates" in plan["sod"]:
        return plan["sod"]["destStates"]
    elif "destStates" in plan:
        return plan["destStates"]
    else:
        raise KeyError("No destStates found in the provided plan JSON.")


def _set_value_or_raise(dict_to_update, key, value):
    if key not in dict_to_update:
        dict_to_update[key] = value
    else:
        if dict_to_update[key] != value:
            raise ValueError("unexpected mismatch", f"{dict_to_update=} {dict_to_update[key]=} {value=}")


def _extract_data(plan: dict):
    """
    extract out the by destination level data for each autopool state

    (lptoken, token) spot price

    (token) safe price

    (lptoken) spot price

    (lptoken) safe price

    (lptoken) autopool owned shares

    (lptoken, token) pool quantity


    # apr data

    incentive, total apr in, total apr out,

    """
    date_info = {"timestamp": plan["timestamp"], "date": pd.to_datetime(plan["timestamp"], unit="s", utc=True)}
    (
        token_safe_prices,
        token_spot_prices,
        pool_spot_prices,
        pool_safe_prices,
        autopool_owned_shares,
        pool_token_quantity,
        destination_incentive_apr,
        destination_total_apr_in,
        destination_total_apr_out,
        destination_fee_and_base_apr,
    ) = [date_info.copy() for _ in range(10)]

    def _extract_price_information_from_solver(dest: dict):

        if "decimals" not in dest:
            decimals = [18 for _ in dest["underlyingTokens"]]
        else:
            decimals = dest["decimals"]
        for i in range(len(decimals)):
            token_address = dest["underlyingTokens"][i]
            underlying_lp_token_address = dest["underlying"]

            _set_value_or_raise(token_safe_prices, token_address, dest["tokenSafePrice"][i])
            _set_value_or_raise(
                token_spot_prices, (underlying_lp_token_address, token_address), dest["tokenSpotPrice"][i]
            )

            _set_value_or_raise(pool_spot_prices, underlying_lp_token_address, dest["spotPrice"])
            _set_value_or_raise(pool_safe_prices, underlying_lp_token_address, dest["safePrice"])

            _set_value_or_raise(autopool_owned_shares, underlying_lp_token_address, dest["ownedShares"] / 1e18)
            _set_value_or_raise(
                pool_token_quantity,
                (underlying_lp_token_address, token_address),
                dest["underlyingTokenAmounts"][i] / 10 ** decimals[i],
            )

            # apr data
            _set_value_or_raise(destination_incentive_apr, underlying_lp_token_address, 100 * dest["incentiveAPR"])
            _set_value_or_raise(destination_total_apr_in, underlying_lp_token_address, 100 * dest["totalAprIn"])
            _set_value_or_raise(destination_total_apr_out, underlying_lp_token_address, 100 * dest["totalAprOut"])

            _set_value_or_raise(
                destination_fee_and_base_apr,
                underlying_lp_token_address,
                100 * (dest["totalAprOut"] - dest["incentiveAPR"]),
            )

    for dest in _get_destination_states(plan):
        _extract_price_information_from_solver(dest)

    return (
        token_safe_prices,
        token_spot_prices,
        pool_spot_prices,
        pool_safe_prices,
        autopool_owned_shares,
        pool_token_quantity,
        destination_incentive_apr,
        destination_total_apr_in,
        destination_total_apr_out,
        destination_fee_and_base_apr,
    )


token_safe_prices_records = []
token_spot_prices_records = []
pool_spot_prices_records = []
pool_safe_prices_records = []
autopool_owned_shares_records = []
pool_token_quantity_records = []
destination_incentive_apr_records = []
destination_total_apr_in_records = []
destination_total_apr_out_records = []
destination_fee_and_base_apr_records = []

for plan in autoETH_plans:
    (
        token_safe_prices,
        token_spot_prices,
        pool_spot_prices,
        pool_safe_prices,
        autopool_owned_shares,
        pool_token_quantity,
        destination_incentive_apr,
        destination_total_apr_in,
        destination_total_apr_out,
        destination_fee_and_base_apr,
    ) = _extract_data(plan)

    token_safe_prices_records.append(token_safe_prices)
    token_spot_prices_records.append(token_spot_prices)
    pool_spot_prices_records.append(pool_spot_prices)
    pool_safe_prices_records.append(pool_safe_prices)
    autopool_owned_shares_records.append(autopool_owned_shares)
    pool_token_quantity_records.append(pool_token_quantity)
    destination_incentive_apr_records.append(destination_incentive_apr)
    destination_total_apr_in_records.append(destination_total_apr_in)
    destination_total_apr_out_records.append(destination_total_apr_out)
    destination_fee_and_base_apr_records.append(destination_fee_and_base_apr)

df_token_safe_prices = pd.DataFrame.from_records(token_safe_prices_records).melt(
    id_vars=["timestamp", "date"], var_name="token", value_name="safe_price"
)


df_pool_spot_prices = pd.DataFrame.from_records(pool_spot_prices_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="spot_price"
)
df_pool_safe_prices = pd.DataFrame.from_records(pool_safe_prices_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="safe_price"
)
df_autopool_owned_shares = pd.DataFrame.from_records(autopool_owned_shares_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="owned_shares"
)
df_destination_incentive_apr = pd.DataFrame.from_records(destination_incentive_apr_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="incentive_apr"
)
df_destination_total_apr_in = pd.DataFrame.from_records(destination_total_apr_in_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="total_apr_in"
)
df_destination_total_apr_out = pd.DataFrame.from_records(destination_total_apr_out_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="total_apr_out"
)
df_destination_fee_and_base_apr = pd.DataFrame.from_records(destination_fee_and_base_apr_records).melt(
    id_vars=["timestamp", "date"], var_name="pool", value_name="fee_and_base_apr"
)


df_token_spot_prices = pd.DataFrame.from_records(token_spot_prices_records)  # not sure
df_pool_token_quantity = pd.DataFrame.from_records(pool_token_quantity_records)  # not sure

df_token_spot_prices = df_token_spot_prices.melt(
    id_vars=["timestamp", "date"], var_name="token", value_name="spot_prices"
)
df_token_spot_prices["pool"] = df_token_spot_prices["token"].apply(lambda x: x[0])
df_token_spot_prices["token"] = df_token_spot_prices["token"].apply(lambda x: x[1])

df_pool_token_quantity = df_pool_token_quantity.melt(
    id_vars=["timestamp", "date"], var_name="token", value_name="quantity"
)
df_pool_token_quantity["pool"] = df_pool_token_quantity["token"].apply(lambda x: x[0])
df_pool_token_quantity["token"] = df_pool_token_quantity["token"].apply(lambda x: x[1])


import functools

vault_dfs = [
    df_pool_spot_prices,
    df_pool_safe_prices,
    df_autopool_owned_shares,
    df_destination_incentive_apr,
    df_destination_total_apr_in,
    df_destination_total_apr_out,
    df_destination_fee_and_base_apr,
]

vault_level_data = functools.reduce(
    lambda left, right: pd.merge(left, right, on=["timestamp", "date", "pool"], how="outer"), vault_dfs
)

token_level_data = df_token_safe_prices.copy()

# todo add token backing here

In [ ]:
vault_token_data = pd.merge(
    left=df_token_spot_prices, right=df_pool_token_quantity, on=["timestamp", "date", "vault", "token"], how="outer"
)
vault_token_data

In [ ]:
long_fee_and_base_apr = df_destination_fee_and_base_apr.melt(
    id_vars=["timestamp", "date"], var_name="destination_address", value_name="apr_value"
)


long_fee_and_base_apr

In [ ]:
break

In [ ]:
def _extract_prices_and_amounts(plan: dict):
    safe_prices = {
        "timestamp": plan["timestamp"],
        "date": pd.to_datetime(plan["timestamp"], unit="s", utc=True),
    }  # token, safe price
    spot_prices = {
        "timestamp": plan["timestamp"],
        "date": pd.to_datetime(plan["timestamp"], unit="s", utc=True),
    }  # (pool, token) spot price

    pool_amounts = {
        "timestamp": plan["timestamp"],
        "date": pd.to_datetime(plan["timestamp"], unit="s", utc=True),
    }  # (pool, token), quantity
    autopool_owned_shares = {
        "timestamp": plan["timestamp"],
        "date": pd.to_datetime(plan["timestamp"], unit="s", utc=True),
    }  # token, quantity

    def _extract_token_amounts_in_destination(dest: dict) -> dict:

        if "decimals" not in dest:
            decimals = [18 for _ in dest["underlyingTokens"]]
        else:
            decimals = dest["decimals"]
        for i in range(len(decimals)):
            token_address = dest["underlyingTokens"][i]
            underlying_lp_token_address = dest["underlying"]
            token_amount = dest["underlyingTokenAmounts"][i] / 10 ** decimals[i]

            ownedShares = dest["ownedShares"] / 10**18
            lp_token_spt_price = dest["safePrice"]
            lp_token_safe_price = dest["spotPrice"]

            token_safe_price = dest["tokenSafePrice"][i]
            token_spot_price = dest["tokenSpotPrice"][i]

            if token_address not in safe_prices:
                safe_prices[token_address] = token_safe_price
            else:
                if safe_prices[token_address] != token_safe_price:
                    raise ValueError("unexpected mismatchs", f"{safe_prices=} {dest=}")

            if (underlying_lp_token_address, token_address) not in spot_prices:
                spot_prices[(underlying_lp_token_address, token_address)] = token_spot_price
            else:
                if spot_prices[(underlying_lp_token_address, token_address)] != token_spot_price:
                    raise ValueError("unexpected mismatch", f"{spot_prices=} {dest=}")

            if (underlying_lp_token_address, token_address) not in pool_amounts:
                pool_amounts[(underlying_lp_token_address, token_address)] = token_amount
            else:
                if pool_amounts[(underlying_lp_token_address, token_address)] != token_amount:
                    raise ValueError("unexpected mismatch ", f"{pool_amounts=} {dest=}")

            if token_address not in autopool_owned_shares:
                autopool_owned_shares[token_address] = ownedShares
            else:
                autopool_owned_shares[token_address] += ownedShares

    if "sod" in plan and "destStates" in plan["sod"]:
        dest_states = plan["sod"]["destStates"]
    elif "destStates" in plan:
        dest_states = plan["destStates"]
    else:
        raise KeyError("No destStates found in the provided plan JSON.")

    for dest in dest_states:
        _extract_token_amounts_in_destination(dest)

    return safe_prices, spot_prices, pool_amounts, autopool_owned_shares


safe_price_records = []
spot_price_records = []
pool_amount_records = []
autopool_amount_records = []

for plan in autoETH_plans:
    safe_prices, spot_prices, pool_amounts, autopool_amounts = _extract_prices_and_amounts(plan)
    safe_price_records.append(safe_prices)
    spot_price_records.append(spot_prices)
    pool_amount_records.append(pool_amounts)
    autopool_amount_records.append(autopool_amounts)


safe_price_df = pd.DataFrame.from_records(safe_price_records).set_index("date").sort_index().drop(columns="timestamp")
spot_price_df = pd.DataFrame.from_records(spot_price_records).set_index("date").sort_index().drop(columns="timestamp")
pool_amount_df = pd.DataFrame.from_records(pool_amount_records).set_index("date").sort_index().drop(columns="timestamp")
autopool_amount_df = (
    pd.DataFrame.from_records(autopool_amount_records).set_index("date").sort_index().drop(columns="timestamp")
)

safe_price_df = safe_price_df.reindex(sorted(safe_price_df.columns), axis=1)
spot_price_df = spot_price_df.reindex(sorted(spot_price_df.columns), axis=1)
pool_amount_df = pool_amount_df.reindex(sorted(pool_amount_df.columns), axis=1)
autopool_shares_df = autopool_amount_df.reindex(sorted(autopool_amount_df.columns), axis=1)

In [ ]:
import plotly.express as px
from plotly.io import templates

templates.default = None
px.area(autopool_amount_df * safe_price_df)

In [ ]:
break

In [ ]:
def normalize_plan(plan: dict) -> list:
    """
    Normalizes the plan JSON into a list of summary dictionaries suitable for conversion
    into a pandas DataFrame.

    This function supports both JSON examples:
      1. When destStates is nested under a "sod" key.
      2. When destStates is available as a top-level key.

    It extracts the following summary fields:
      - name: The pool name.
      - vault address: The pool's address.
      - incentiveApr: The incentive APR from the pool.
      - ownedShares: Normalized using the decimals value (i.e. int(ownedShares) / (10 ** decimals)).
          If the decimals field is missing, a default of 18 is used.
      - compositeReturn: Computed as the average of totalAprIn and totalAprOut.
      - pricePerShare: Taken from the safePrice field.

    Parameters:
        plan (dict): The plan JSON data.

    Returns:
        list: A list of dictionaries with normalized summary fields.
    """
    if "sod" in plan and "destStates" in plan["sod"]:
        dest_states = plan["sod"]["destStates"]
    elif "destStates" in plan:
        dest_states = plan["destStates"]
    else:
        raise KeyError("No destStates found in the provided plan JSON.")

    def _extract_fields_to_perserve_without_transformation(dest: dict) -> dict:
        fields_to_preserve_as_is = [
            "snapshotTimestamp",
            "name",
            "address",  # Destination Vault Address
            "poolType",
            "pool",
            "underlying",
            "flipFlag",
            "discountViolationAddFlag",
            "discountViolationTrim1Flag",
            "discountViolationTrim2Flag",
        ]
        return {k: dest[k] for k in fields_to_preserve_as_is}

    def _extract_token_amounts_in_destination(dest: dict) -> dict:

        if "decimals" not in dest:
            decimals = [18 for _ in dest["underlyingTokens"]]
        else:
            decimals = dest["decimals"]

        pool_details = {}
        for i in range(len(decimals)):
            # details on the pool itselef
            pool_details[f'{dest["underlyingTokens"][i]}_amount'] = float(
                dest["underlyingTokenAmounts"][i] / 10 ** decimals[i]
            )
            pool_details[f'{dest["underlyingTokens"][i]}_spot_value'] = (
                pool_details[f'{dest["underlyingTokens"][i]}_amount'] * dest["tokenSpotPrice"][i]
            )
            pool_details[f'{dest["underlyingTokens"][i]}_safe_value'] = (
                pool_details[f'{dest["underlyingTokens"][i]}_amount'] * dest["tokenSafePrice"][i]
            )

        return pool_details

    def _extract_pool_level_data(dest: dict) -> dict:
        autopool_ownership_data = {"ownedShares": int(dest["ownedShares"]) / 1e18}
        autopool_ownership_data["autopoolSafeValueInDestination"] = (
            autopool_ownership_data["ownedShares"] * dest["safePrice"]
        )
        autopool_ownership_data["autopoolSpotValueInDestination"] = (
            autopool_ownership_data["ownedShares"] * dest["spotPrice"]
        )

        return autopool_ownership_data

    def _extract_destination_apr_values(dest: dict) -> dict:
        apr_fields_to_upscale = [
            "totalAprIn",
            "totalAprOut",
            "incentiveAPR",
        ]
        apr_data = {k: dest[k] * 100 for k in apr_fields_to_upscale}

        apr_data["fee_plus_base_apr"] = apr_data["totalAprOut"] - apr_data["incentiveAPR"]

        return apr_data

    autopool_data = []
    destination_data = []
    for dest in dest_states:
        identity_fields = _extract_fields_to_perserve_without_transformation(dest)

        autopool_ownership_data = _extract_pool_level_data(dest)
        autopool_data.append({**identity_fields, **autopool_ownership_data})
        token_amounts_in_pool = _extract_token_amounts_in_destination(dest)

        destination_data.append(
            {
                "solverTimestamp": plan["timestamp"],
                **identity_fields,
                **token_amounts_in_pool,
            }
        )

    return autopool_data, destination_data


autopool_data, destination_data = normalize_plan(autoETH_plans[-1])
pd.DataFrame(destination_data)

In [ ]:
# allocation over time df

In [ ]:
autoETH_plans

In [ ]:
autoETH_plans[-1]